# 0. Environment Setup

In [65]:
import torch
import random
import numpy as np
from sklearn.base import RegressorMixin, BaseEstimator

# 1. My Linear regression implementation

In [66]:
class myLinearRegression(RegressorMixin, BaseEstimator):
    def __init__(self, lr, batch_size, n_iter):
        self.lr = lr
        self._batch_size = batch_size
        self._n_iter = n_iter
        self._w = None
        self._b = None
    
    def fit(self, Xs, ys):
        num_samples, num_features = Xs.shape
        self._w = torch.normal(0, 1, (num_features, 1), requires_grad=True)
        self._b = torch.tensor(0.0, requires_grad=True)
        for _ in range(self._n_iter):
            for X_batch, y_batch in self.get_batch(Xs, ys):
                y_pred = self.predict(X_batch)
                self.backward(y_pred, y_batch)
                self._w.grad.zero_()
                self._b.grad.zero_()
        
    def predict(self, Xs):
        return Xs @ self._w + self._b
    
    def backward(self, y_pred, ys):
        loss = ((y_pred - ys) ** 2).mean()
        loss.backward()
        self._w.data -= self.lr * self._w.grad
        self._b.data -= self.lr * self._b.grad
        
    def get_batch(self, Xs, ys):
        num_samples = Xs.shape[0]
        indices = list(range(num_samples))
        random.shuffle(indices)
        for i in range(0, num_samples, self._batch_size):
            batch_indices = indices[i:i+self._batch_size]
            yield Xs[batch_indices], ys[batch_indices]
        

In [67]:
# Test data
torch.manual_seed(42)
Xs = torch.normal(0, 1, (1000, 13))
w_real = torch.tensor([1, 1, 4, 5, 1, 4, 1, 9, 1, 9, 8, 1, 0], dtype=torch.float32).reshape(-1, 1)
b_real = 67
ys = Xs @ w_real + b_real + torch.normal(0, 1, (1000, 1)) * 0.01
print(f"Size of Xs is {Xs.shape}, size of ys is {ys.shape}")
print(f"First 5 rows of Xs: {Xs[:5]}")
print(f"First 5 rows of ys: {ys[:5]}")


Size of Xs is torch.Size([1000, 13]), size of ys is torch.Size([1000, 1])
First 5 rows of Xs: tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431, -1.6047,
         -0.7521,  1.6487, -0.3925, -1.4036, -0.7279],
        [-0.5594, -0.7688,  0.7624,  1.6423, -0.1596, -0.4974,  0.4396, -0.7581,
          1.0783,  0.8008,  1.6806,  1.2791,  1.2964],
        [ 0.6105,  1.3347, -0.2316,  0.0418, -0.2516,  0.8599, -1.3847, -0.8712,
         -0.2234,  1.7174,  0.3189, -0.4245,  0.3057],
        [-0.7746, -1.5576,  0.9956, -0.8798, -0.6011, -1.2742,  2.1228, -1.2347,
         -0.4879, -0.9138, -0.6581,  0.0780,  0.5258],
        [-0.4880,  1.1914, -0.8140, -0.7360, -1.4032,  0.0360, -0.0635,  0.6756,
         -0.0978,  1.8446, -1.1845,  1.3835,  1.4451]])
First 5 rows of ys: tensor([[54.2716],
        [91.4124],
        [79.5296],
        [35.6644],
        [73.9408]])


In [68]:
def tensor2str(t):
    if isinstance(t, torch.Tensor):
        return np.array2string(t.detach().flatten().numpy(),
                               formatter={'float_kind': '{:.4f}'.format},
                               separator=', ')
    else: 
        return t

myLR = myLinearRegression(lr=0.01, batch_size=32, n_iter=1000)
myLR.fit(Xs, ys)
print(f"Learned weights: {tensor2str(myLR._w)}\nreal weights: {tensor2str(w_real)}")
print("The difference between learned weights and real weights is:\n" + tensor2str((myLR._w.reshape(-1).detach().flatten() - w_real.reshape(-1)).abs()))
print(f"Learned bias: {tensor2str(myLR._b)}, real bias: {tensor2str(b_real)}")
print(f"First 5 predictions and real values:")
for i in range(5):
    print(f"Prediction: {tensor2str(myLR.predict(Xs[i:i+1]))}, Real: {tensor2str(ys[i])}")

Learned weights: [0.9994, 1.0002, 4.0005, 5.0003, 1.0000, 4.0002, 0.9999, 9.0006, 1.0003,
 8.9997, 8.0000, 1.0000, 0.0003]
real weights: [1.0000, 1.0000, 4.0000, 5.0000, 1.0000, 4.0000, 1.0000, 9.0000, 1.0000,
 9.0000, 8.0000, 1.0000, 0.0000]
The difference between learned weights and real weights is:
[0.0006, 0.0002, 0.0005, 0.0003, 0.0000, 0.0002, 0.0001, 0.0006, 0.0003,
 0.0003, 0.0000, 0.0000, 0.0003]
Learned bias: [66.9999], real bias: 67
First 5 predictions and real values:
Prediction: [54.2843], Real: [54.2716]
Prediction: [91.4108], Real: [91.4124]
Prediction: [79.5482], Real: [79.5296]
Prediction: [35.6644], Real: [35.6644]
Prediction: [73.9361], Real: [73.9408]


# 2. Linear Regression with nn of PyTorch

In [69]:
from torch.utils import data
from torch import nn

batch_size = 32
dataset = data.TensorDataset(Xs, ys)
data_iter = data.DataLoader(dataset, batch_size, shuffle=True)

net = nn.Linear(Xs.shape[1], 1)
net.weight.data.normal_(0, 1)
net.bias.data.fill_(0)

loss = nn.MSELoss()

trainer = torch.optim.SGD(net.parameters(), lr=0.01)

n_iter = 1000
for i in range(n_iter):
    for X_batch, y_batch in data_iter:
        trainer.zero_grad()
        y_pred = net(X_batch)
        l = loss(y_pred, y_batch)
        l.backward()
        trainer.step()
    if (i + 1) % 100 == 0:
        l = loss(net(Xs), ys)
        print(f'epoch {i + 1}, loss {l:f}')

epoch 100, loss 0.000100
epoch 200, loss 0.000100
epoch 300, loss 0.000100
epoch 400, loss 0.000100
epoch 500, loss 0.000100
epoch 600, loss 0.000100
epoch 700, loss 0.000100
epoch 800, loss 0.000100
epoch 900, loss 0.000100
epoch 1000, loss 0.000100


In [70]:
print(f"Learned weights: {tensor2str(net.weight)}\nreal weights: {tensor2str(w_real)}")
print("The difference between learned weights and real weights is:\n" + tensor2str((net.weight.reshape(-1).detach().flatten() - w_real.reshape(-1)).abs()))
print(f"Learned bias: {tensor2str(net.bias)}, real bias: {tensor2str(b_real)}")
print(f"First 5 predictions and real values:")
for i in range(5):
    net.eval()
    print(f"Prediction: {tensor2str(net(Xs[i:i+1]))}, Real: {tensor2str(ys[i])}")

Learned weights: [0.9994, 1.0003, 4.0004, 5.0003, 0.9998, 4.0004, 1.0001, 9.0005, 1.0002,
 8.9998, 7.9999, 1.0000, 0.0002]
real weights: [1.0000, 1.0000, 4.0000, 5.0000, 1.0000, 4.0000, 1.0000, 9.0000, 1.0000,
 9.0000, 8.0000, 1.0000, 0.0000]
The difference between learned weights and real weights is:
[0.0006, 0.0003, 0.0004, 0.0003, 0.0002, 0.0004, 0.0001, 0.0005, 0.0002,
 0.0002, 0.0001, 0.0000, 0.0002]
Learned bias: [66.9999], real bias: 67
First 5 predictions and real values:
Prediction: [54.2846], Real: [54.2716]
Prediction: [91.4105], Real: [91.4124]
Prediction: [79.5482], Real: [79.5296]
Prediction: [35.6648], Real: [35.6644]
Prediction: [73.9365], Real: [73.9408]
